# Flux Processing Chain

Post-processing of Level-1 eddy covariance fluxes (EddyPro FLUXNET output) through the
levels of the Swiss FluxNet workflow:

| Level | What it does | Function |
|---|---|---|
| L2 | EddyPro quality-flag expansion + QCF | `run_level2` |
| L3.1 | Storage correction (`FC` -> `NEE`) | `run_level31` |
| L3.2 | Outlier removal | `make_level32_detector` + `run_level32` |
| L3.3 | USTAR filtering | `run_level33_constant_ustar` |
| L4.1 | Gap-filling | `run_level41_rf` / `_xgb` / `_mds` |

Each level is a plain function that takes the `FluxLevelData` container and returns a
**new** one. Nothing is mutated in place, so you can branch: run L4.1 three different
ways from the same L3.3 state without repeating the upstream levels.

> **Note on the API.** Earlier versions of this notebook used a single stateful
> `FluxProcessingChain` object. That class was replaced in v0.91.0 by the composable
> functions used here. The reporting it offered lives in
> `diive.flux.fluxprocessingchain` as `report_*` functions that take `data`.

## Imports

In [ ]:
import diive as dv
from diive.configs.exampledata import load_exampledata_parquet_lae_level1_30MIN
from diive.core.ml.feature_engineer import FeatureEngineer
from diive.flux.fluxprocessingchain import (
    init_flux_data,
    make_level32_detector,
    run_level2,
    run_level31,
    run_level32,
    run_level33_constant_ustar,
    run_level41_mds,
    run_level41_rf,
    run_level41_xgb,
    # Reporting over the finished chain
    gapfilled_variables,
    plot_feature_ranks_per_year,
    plot_gapfilled_cumulative,
    plot_mds_gapfilling_qualities,
    report_gapfilling_feature_importances,
    report_gapfilling_model_scores,
    report_gapfilling_poolyears,
    report_gapfilling_variables,
    report_traintest_details,
    report_traintest_model_scores,
)

## Load data

The bundled example is one site's Level-1 data. To use your own EddyPro FLUXNET output,
replace this cell with `dv.ReadFileType(...)` or `dv.load_parquet(...)`.

`init_flux_data` computes `SW_IN_POT`, `DAYTIME` and `NIGHTTIME` itself and raises if any
of them already exists, so drop them first.

In [ ]:
df = load_exampledata_parquet_lae_level1_30MIN()
df = df.loc['2024-07':'2024-07']  # one month keeps the notebook quick
df = df.drop(columns=[c for c in ('SW_IN_POT', 'DAYTIME', 'NIGHTTIME') if c in df.columns])
df.head()

## Step 1 - initialise the container

Calculates potential radiation, derives day/night flags and freezes the site metadata that
every later level reads.

`daytime_accept_qcf_below=2` keeps QCF=0 (all tests pass) and QCF=1 (soft warnings).
Set it to 1 to accept only the strictest quality.

In [ ]:
data = init_flux_data(
    df=df,
    fluxcol='FC',
    site_lat=47.41887,  # CH-HON
    site_lon=8.491318,
    utc_offset=1,
    nighttime_threshold=20,  # W m-2 potential radiation below this = nighttime
    daytime_accept_qcf_below=2,
    nighttime_accept_qcf_below=2,
)
print(data)

## Step 2 - Level 2: quality flag expansion

Each EddyPro test is switched on with a config dict. Omit a test to skip it.

In [ ]:
data = run_level2(
    data,
    ssitc={'apply': True, 'setflag_timeperiod': None},
    gas_completeness={'apply': True},
    spectral_correction_factor={'apply': True},
    raw_data_screening_vm97={
        'apply': True,
        'spikes': True, 'amplitude': False, 'dropout': True,
        'abslim': False, 'skewkurt_hf': False, 'skewkurt_sf': False,
        'discont_hf': False, 'discont_sf': False,
    },
)
print(f'After L2: {data.filteredseries.dropna().count()} valid records')

In [ ]:
data.levels.level2_qcf.showplot_qcf_heatmaps()

In [ ]:
data.levels.level2_qcf.report_qcf_flags()

## Step 3 - Level 3.1: storage correction

Adds the storage term to the flux (`NEE = FC + SC_SINGLE`). For H / LE without a storage
profile, pass `set_storage_to_zero=True` instead.

In [ ]:
data = run_level31(data, gapfill_storage_term=True, set_storage_to_zero=False)
print(f'After L3.1: {data.filteredseries.dropna().count()} valid records '
      f'(col: {data.filteredseries.name})')

## Step 4 - Level 3.2: outlier removal

`make_level32_detector` hands back a detector wired to the current state. Add as many
`flag_outliers_*` + `addflag()` pairs as you need, then pass it to `run_level32`.

In [ ]:
data, sod = make_level32_detector(data)

sod.flag_outliers_hampel_test(
    window_length=48 * 13,  # 13-day rolling window
    n_sigma_daytime=5.5,
    n_sigma_nighttime=5.5,
    use_differencing=True,
    separate_day_night=True,
    showplot=False,
    verbose=True,
    repeat=True,
)
sod.addflag()

data = run_level32(data, outlier_detector=sod)
print(f'After L3.2: {data.filteredseries.dropna().count()} valid records')

In [ ]:
data.levels.level32_qcf.showplot_qcf_heatmaps()

## Step 5 - Level 3.3: USTAR filtering

One threshold per scenario. Pass several for a sensitivity analysis, e.g.
`thresholds=[0.10, 0.18, 0.25]` with `threshold_labels=['CUT_16', 'CUT_50', 'CUT_84']`.
For energy fluxes (H, LE) use `thresholds=[0], threshold_labels=['CUT_NONE']`.

After this call `data.filteredseries` is `None` - there is no single filtered series
across scenarios any more. Read them per scenario instead.

In [ ]:
data = run_level33_constant_ustar(
    data,
    thresholds=[0.30],  # m s-1, site-specific
    threshold_labels=['CUT_50'],
    showplot=False,
    verbose=True,
)

flux_l33 = data.levels.filteredseries_level33_qcf['CUT_50']
flux_l33_hq = data.levels.filteredseries_level33_hq['CUT_50']  # QCF=0 only
print(f'After L3.3 (CUT_50): {flux_l33.dropna().count()} accepted | '
      f'{flux_l33_hq.dropna().count()} high-quality')

### Gaps before gap-filling

`data.gap_stats(level)` works at any level and returns one `GapStats` per scenario.

In [ ]:
for scen, gs in data.gap_stats('L3.3').items():
    gs.report()

## Step 6 - feature engineering for the ML gap-fillers

Build the engineer once and hand the same instance to both Random Forest and XGBoost.
All feature columns must exist in `data.full_df` - use `add_driver(data, series)` to put
one there. `target_col` is a placeholder here; L4.1 applies the engineer to predictors only.

In [ ]:
FEATURES = ['TA_T1_47_1_gfXG', 'SW_IN_T1_47_1_gfXG', 'VPD_T1_47_1_gfXG']

engineer = FeatureEngineer(
    target_col='_target_',
    features_lag=[-2, 2],
    features_lag_stepsize=1,
    features_rolling=[2, 4, 12, 24, 48],
    features_rolling_stats=['median', 'min', 'max', 'std'],
    features_ema=[6, 12, 24, 48],
    vectorize_timestamps=True,
    add_continuous_record_number=True,
    sanitize_timestamp=True,
    verbose=1,
)

## Step 7 - Level 4.1: gap-filling

Three methods, all branching from the same L3.3 state. The settings below are small so the
notebook runs quickly; for production raise `n_estimators` and `max_depth`.

In [ ]:
data = run_level41_rf(
    data,
    features=FEATURES,
    engineer=engineer,
    reduce_features=True,  # SHAP-based feature selection
    verbose=1,
    n_estimators=9,        # production: >= 350
    max_depth=None,        # production: >= 15
    random_state=42,
    n_jobs=-1,
    shap_max_rows=5000,
)

In [ ]:
data = run_level41_xgb(
    data,
    features=FEATURES,
    engineer=engineer,
    reduce_features=True,
    verbose=1,
    n_estimators=99,       # production: >= 350
    max_depth=6,
    learning_rate=0.05,
    early_stopping_rounds=10,
    random_state=42,
    n_jobs=-1,
    shap_max_rows=5000,
)

MDS needs its drivers in `data.full_df`, and **VPD in kPa** (not hPa).

In [ ]:
data = run_level41_mds(
    data,
    swin='SW_IN_T1_47_1_gfXG',
    ta='TA_T1_47_1_gfXG',
    vpd='VPD_T1_47_1_gfXG',  # kPa
    ta_tol=2.5,
    vpd_tol=0.5,
)

## Step 8 - what came out

`gapfilled_cols()` gives the output column per method and scenario;
`nongapfilled_cols()` gives the measured column each one filled.

In [ ]:
print(data.gapfilled_cols())
print(data.nongapfilled_cols())

report_gapfilling_variables(data)

In [ ]:
gapfilled_variables(data).describe()

## Step 9 - model reporting

Held-out test scores first. These come from a random train/test split of the complete
rows, which is the split that reproduces the gap-filling task. MDS trains no model, so it
has no such split and is reported as unavailable rather than raising.

Pass `outpath='...'` to any of these to also write one CSV per method and scenario.

In [ ]:
report_traintest_model_scores(data)

In [ ]:
report_traintest_details(data)

In-sample scores are optimistically biased - compare them against the held-out scores above.

In [ ]:
report_gapfilling_model_scores(data)

Which years each year's model was trained on. The long-term gap-fillers pool neighbouring
years; MDS does not pool.

In [ ]:
report_gapfilling_poolyears(data)

Per-year SHAP feature importances for the ML methods.

In [ ]:
report_gapfilling_feature_importances(data)

In [ ]:
plot_feature_ranks_per_year(data)

## Step 10 - result plots

Measured vs gap-filled, one panel per method:

In [ ]:
data.plot_gapfilled_heatmaps(ustar_scenario='CUT_50')

All methods overlaid on one cumulative curve, for a direct comparison:

In [ ]:
UMOL_TO_GC = 12.011 * 1e-6 * 1800  # umol CO2 m-2 s-1 over 30 min -> gC m-2

data.plot_cumulative_comparison(
    ustar_scenario='CUT_50',
    conv_factor=UMOL_TO_GC,
    units='gC m-2',
)

One cumulative per gap-filled variable over the whole record. With several years of data,
`per_year=True` draws each year as its own line with a multi-year reference band.

In [ ]:
plot_gapfilled_cumulative(
    data,
    gain=UMOL_TO_GC,
    units='gC m-2',
    per_year=False,
)

MDS fill quality - which step of the cascade filled each record:

In [ ]:
plot_mds_gapfilling_qualities(data)

## Step 11 - export

`merged_df()` puts the untouched input columns and everything the chain produced into one
dataframe. Nothing in the input is overwritten; the chain only adds columns.

In [ ]:
out_df = data.merged_df()
out_df.shape

In [ ]:
# dv.save_parquet(data=out_df, filename='fluxprocessingchain_results', outpath='.')